# 05 — Audio with TorchAudio: Loading, Spectrograms, Augmentations, and Model Patterns

Goal: build audio pipelines and models using TorchAudio.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. TorchAudio availability

In [ ]:

try:
    import torchaudio
    import torchaudio.transforms as AT
    print("torchaudio:", torchaudio.__version__)
except Exception as e:
    print("torchaudio not available:", e)

## 2. Audio loading pattern

`torchaudio.load` returns:
- waveform: [channels, time]
- sample_rate

In [ ]:

audio_overview = r'''
import torchaudio
waveform, sr = torchaudio.load("file.wav")
if waveform.size(0) > 1:
    waveform = waveform.mean(dim=0, keepdim=True)  # mono
'''
print(audio_overview)

## 3. Spectrograms and mel spectrograms

In [ ]:

spec_template = r'''
import torchaudio.transforms as AT
mel = AT.MelSpectrogram(sample_rate=16000, n_fft=400, hop_length=160, n_mels=80)
x_mel = mel(waveform)  # [C, n_mels, frames]
'''
print(spec_template)

## 4. Audio classification model pattern (CNN on mel)

In [ ]:

import torch, torch.nn as nn
import torch.nn.functional as F

class AudioCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1)),
        )
        self.fc = nn.Linear(32, num_classes)
    def forward(self, x):
        x = self.net(x).squeeze(-1).squeeze(-1)
        return self.fc(x)

m = AudioCNN().to(device)
m(torch.randn(4,1,80,100, device=device)).shape